In [1]:
import numpy as np
import json
from classy import Class
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import cosmoprimo
import os
import pandas as pd

from FishLSS.fisherForecast import fisherForecast
from FishLSS.experiment import experiment

In [ ]:
# filename for example survey
bfn = 'Euclid_corr'
# bfn_no_subscripts = bfn.replace('_14bins', '')
# bfn_no_subscripts = bfn.replace('_28bins', '')
# bfn_no_subscripts = bfn.replace('_', '')
bd = '/home/adrien/PDM/code/PDM2026_wsl/derivatives/'

log_path = '/home/adrien/miniconda3/envs/fishlss/lib/python3.11/site-packages/FishLSS/bao_recon/log.npy'

In [19]:
f = open('../derivatives/output/'+bfn+'/summary.json')
summary = json.load(f)

In [20]:
print('summary = ')
for keys in summary.keys():
    print(keys, summary[keys])

summary = 
Forecast name DESI_ext_noQSO
Edges of redshift bins [0.15, 0.42, 0.6, 0.8, 1.05, 1.6]
Centers of redshift bins [0.285, 0.51, 0.7, 0.925, 1.3250000000000002]
Linear Eulerian bias in each bin [1.6, 2.0, 2.0, 2.0, 1.2]
Number density in each bin [0.0007650000000000001, 0.0006, 0.0006, 0.00045, 0.00027999999999999987]
fsky 0.41
CLASS default parameters {'output': 'tCl lCl mPk', 'l_max_scalars': 2000, 'lensing': 'yes', 'non linear': 'halofit', 'A_s': 2.083e-09, 'n_s': 0.9649, 'alpha_s': 0.0, 'h': 0.6736, 'N_ur': 2.0328, 'N_ncdm': 1, 'm_ncdm': 0.06, 'tau_reio': 0.0544, 'omega_b': 0.02237, 'omega_cdm': 0.12, 'Omega_k': 0.0, 'P_k_max_h/Mpc': 2.0, 'z_pk': '0.0,6'}


In [21]:
params = summary['CLASS default parameters']
cosmo = Class() 
cosmo.set(params) 
cosmo.compute() 

In [22]:
# load fiducial linear bias/number density from table
zs, bs, ns = np.genfromtxt('/home/adrien/PDM/code/PDM2026_wsl/FishLSS_script/' + bfn + '.txt').T

# assume zs spans the full survey
ze = summary['Edges of redshift bins']
zmin = ze[0]
zmax = ze[-1]

z_centers = summary['Centers of redshift bins']

# interpolate
b = interp1d(zs,bs)
n = interp1d(zs,ns)

nbins = len(ze)-1
fsky = summary['fsky']

exp = experiment(zedges=np.array(ze), nbins=nbins, fsky=fsky, b=b, n=n)

name = summary['Forecast name']

In [23]:
forecast = fisherForecast(experiment=exp,cosmo=cosmo,name=name,basedir=bd)

In [24]:
basis = np.array(['alpha_perp','alpha_parallel','b'])

# set recon = True, so that we perform BAO reconstruction when computing the power spectrum
forecast.recon = True

# set the "marginalized parameters", aka the derivatives, to be [alpha's, linear b]
forecast.free_params = basis

derivs = forecast.load_derivatives(basis) # load the pre computed derivatives

In [25]:
F = lambda i: forecast.gen_fisher(basis, 100, derivatives=derivs, zbins=np.array([i]))
Fs = [F(i) for i in range(nbins)]
Fs = np.array(Fs)

In [26]:
Finvs = [np.linalg.inv(Fs[i]) for i in range(nbins)]
saperp = [np.sqrt(Finvs[i][0,0]) for i in range(nbins)]
saparr = [np.sqrt(Finvs[i][1,1]) for i in range(nbins)]

### Save $\alpha_\perp$ $\alpha_\parallel$ errors

#### fcts

In [2]:
def create_DESI_fid_data(redshifts, DESI_style=False):
    cosmo_planck = cosmoprimo.fiducial.DESI()
    bkg = cosmo_planck.get_background(engine="class")
    thermo = cosmo_planck.get_thermodynamics()
    rdrag = thermo.rs_drag  # no parentheses needed, it's a property

    DM = bkg.comoving_angular_distance(redshifts)
    DH = 1 / bkg.efunc(redshifts) * 2997.92  # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV = (redshifts * DM**2 * DH)**(1/3)
    DM_rd = DM / rdrag
    DH_rd = DH / rdrag
    DV_rd = DV / rdrag

    data_typ = ['DH_over_rs', 'DM_over_rs', 'DV_over_rs']

    fake_data = []
    if isinstance(DESI_style, list):
        for i in range(len(redshifts)):
            if DESI_style[i]:
                line = [f"{float(redshifts[i]):.8e}", f"{DV_rd[i]:.8e}", data_typ[2]]
                fake_data.append(line)
            else:
                line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
                fake_data.append(line)
                line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
                fake_data.append(line)
    elif isinstance(DESI_style, bool) and DESI_style:
        for i in range(1, len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

        line = [f"{float(redshifts[0]):.8e}", f"{DV_rd[0]:.8e}", data_typ[2]]
        fake_data.insert(0, line)
    else:
        for i in range(len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

    return np.array(fake_data)

def save_mean_data(fake_data, folder, filename, overwrite=False):
    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, fake_data, fmt="%s")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, fake_data, fmt="%s")

def cov_from_Fisher_with_units(redshifts, Fish_inv, nbins, with_units=True, DESI_style=False):
    cov_mat = np.zeros((2*nbins, 2*nbins))
    mean = create_DESI_fid_data(redshifts, DESI_style=False)

    if not with_units:
        mean[:,1] = 1.0


    for i in range(nbins):
        if isinstance(DESI_style, list):
            if i == 0 : cov_mat = np.zeros((2*nbins-len([a for a in DESI_style if a]), 2*nbins-len([a for a in DESI_style if a])))
            if DESI_style[i]:
                sig2_DM = Fish_inv[i][0,0] * float(mean[2*i,1])**2
                sig2_DH = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
                DV = (redshifts[i] * float(mean[2*i,1])**2 * float(mean[2*i+1,1]))**(1/3)
                sig2_DV = DV**2 * ( (2/3)**2 * (sig2_DM / float(mean[2*i,1])**2) + (1/3)**2 * (sig2_DH / float(mean[2*i+1,1])**2) )
                cov_mat[i,i] = sig2_DV
            else:
                cov_mat[2*i-1,2*i-1] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
                cov_mat[2*i,2*i] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
                cov_mat[2*i-1,2*i] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
                cov_mat[2*i,2*i-1] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])

        elif isinstance(DESI_style, bool) and DESI_style:
            if i == 0:
                sig2_DM = Fish_inv[i][0,0] * float(mean[0,1])**2
                sig2_DH = Fish_inv[i][1,1] * float(mean[1,1])**2
                DV = (redshifts[i] * float(mean[0,1])**2 * float(mean[1,1]))**(1/3)
                sig2_DV = DV**2 * ( (2/3)**2 * (sig2_DM / float(mean[0,1])**2) + (1/3)**2 * (sig2_DH / float(mean[1,1])**2) )
                cov_mat = np.zeros((2*nbins-1, 2*nbins-1))
                cov_mat[0,0] = sig2_DV
            else:
                cov_mat[2*i-1,2*i-1] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
                cov_mat[2*i,2*i] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
                cov_mat[2*i-1,2*i] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
                cov_mat[2*i,2*i-1] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])

        else:
            cov_mat[2*i,2*i] = Fish_inv[i][0,0] * float(mean[2*i,1])**2
            cov_mat[2*i+1,2*i+1] = Fish_inv[i][1,1] * float(mean[2*i+1,1])**2
            cov_mat[2*i,2*i+1] = Fish_inv[i][0,1] * float(mean[2*i,1]) * float(mean[2*i+1,1])
            cov_mat[2*i+1,2*i] = Fish_inv[i][1,0] * float(mean[2*i+1,1]) * float(mean[2*i,1])
    
    return cov_mat

def save_a_cov_mat(cov_mat, folder, filename, overwrite=False):

    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, cov_mat, fmt="%.8e")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, cov_mat, fmt="%.8e")

#### saving cov mat

In [28]:
overwrite = False

saving cov mat mean values and redshifts

In [29]:
folder_to_save = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/' + bfn
if not os.path.exists(folder_to_save):
    os.makedirs(folder_to_save)

save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, with_units=False, DESI_style=False),
                folder_to_save , '/cov_alpha.txt', overwrite=overwrite)
save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, with_units=True, DESI_style=False), 
                folder_to_save , '/cov_DM_DH.txt', overwrite=overwrite)

f = folder_to_save + '/redshifts.txt'
if overwrite:
    print("Overwriting file:", f)
    np.savetxt(f, z_centers, fmt="%.8e")
else:
    if not os.path.exists(f):
        np.savetxt(f, z_centers, fmt="%.8e")
    else:
        print("File already exists:", f)

In [30]:
data = create_DESI_fid_data(z_centers, DESI_style=False)
save_mean_data(data, folder_to_save, '/mean_LCDM_fid.txt', overwrite=overwrite)

In [31]:
f = open('../derivatives/output/'+bfn+'/summary.json')
summary = json.load(f)
# save the summary.json file in the cov_mat folder
summary_file_path = folder_to_save + '/summary.json'
if overwrite:
    print("Overwriting file:", summary_file_path)
    with open(summary_file_path, 'w') as f:
        json.dump(summary, f, indent=4)
else:
    if not os.path.exists(summary_file_path):
        with open(summary_file_path, 'w') as f:
            json.dump(summary, f, indent=4)
    else:
        print("File already exists:", summary_file_path)

DESI style files

In [16]:
overwrite = False

In [17]:
# Desi-style covariance matrix
# mask = [1, 1]
if z_centers[0] < 0.35:
    mask = True
    save_a_cov_mat(cov_from_Fisher_with_units(z_centers, Finvs, nbins, with_units=True, DESI_style=mask), 
                        folder_to_save , '/cov_DM_DH_DESIstyle.txt', overwrite=overwrite)

    # if z_centers[0] < 0.35:
    data_DESI = create_DESI_fid_data(z_centers, DESI_style=mask)
    save_mean_data(data_DESI, folder_to_save, '/mean_LCDM_fid_DESIstyle.txt', overwrite=overwrite)

## combining forecast fcts

In [3]:
def combine_cov_mat(cov1, cov2):
    """
    Combine two covariance matrices into a block diagonal matrix.
    
    Parameters:
    cov1 : np.ndarray
        First covariance matrix.
    cov2 : np.ndarray
        Second covariance matrix.
        
    Returns:
    np.ndarray
        Combined block diagonal covariance matrix.
    """
    if cov1.ndim != 2 or cov2.ndim != 2:
        raise ValueError("Both cov1 and cov2 must be 2D arrays.")

    return np.block([[cov1, np.zeros((cov1.shape[0], cov2.shape[1]))],
                     [np.zeros((cov2.shape[0], cov1.shape[1])), cov2]])

def combine_forecast(folder_list):
    '''
    Recursively combine the forecasts from multiple folders into a single one
    '''
    if len(folder_list) < 2:
        raise ValueError("At least two folders are required to combine forecasts.")
    elif len(folder_list) == 2:
        return combine(folder_list[0], folder_list[1])
    else:
        pair = combine_forecast(folder_list[1:])
        return combine(folder_list[0], pair)

def combine(folder1, folder2):
    #combine covariance matrices
    cov1 = np.loadtxt(folder1 + '/cov_DM_DH.txt')
    cov2 = np.loadtxt(folder2 + '/cov_DM_DH.txt')
    combined_cov_DM_DH = combine_cov_mat(cov1, cov2)
    cov1 = np.loadtxt(folder1 + '/cov_alpha.txt')
    cov2 = np.loadtxt(folder2 + '/cov_alpha.txt')
    combined_cov_alpha = combine_cov_mat(cov1, cov2)

    # DV/rd can only be in the first folder:
    combined_cov_DM_DH_DESIstyle = None
    if os.path.exists(folder1 + '/cov_DM_DH_DESIstyle.txt'):
        cov1 = np.loadtxt(folder1 + '/cov_DM_DH_DESIstyle.txt')
        cov2 = np.loadtxt(folder2 + '/cov_DM_DH.txt')
        combined_cov_DM_DH_DESIstyle = combine_cov_mat(cov1, cov2)

    # combine redshifts
    z1 = np.loadtxt(folder1 + '/redshifts.txt')
    if z1.ndim == 0: z1 = np.array([z1])
    z2 = np.loadtxt(folder2 + '/redshifts.txt')
    if z2.ndim == 0: z2 = np.array([z2])
    combined_z = np.concatenate((z1, z2))

    # combine mean data
    mean1 = pd.read_csv(
        folder1 + '/mean_LCDM_fid.txt',
        sep=r"\s+",
        header=None,
    )
    mean2 = pd.read_csv(
        folder2 + '/mean_LCDM_fid.txt',
        sep=r"\s+",
        header=None,
    )
    combined_mean = pd.concat([mean1, mean2], ignore_index=True)

    combined_mean_DESI = None
    if os.path.exists(folder1 + '/mean_LCDM_fid_DESIstyle.txt'):
        mean1_DESI = pd.read_csv(
            folder1 + '/mean_LCDM_fid_DESIstyle.txt',
            sep=r"\s+",
            header=None,
        )
        mean2_DESI = pd.read_csv(
            folder2 + '/mean_LCDM_fid.txt',
            sep=r"\s+",
            header=None,
        )
        combined_mean_DESI = pd.concat([mean1_DESI, mean2_DESI], ignore_index=True)

    return combined_cov_DM_DH, combined_cov_alpha, combined_cov_DM_DH_DESIstyle, combined_z, combined_mean, combined_mean_DESI

def remove_z(folder, redshifts):
    '''
    Remove given redshifts from the files.
    '''
    mean = pd.read_csv(folder + '/mean_LCDM_fid.txt', sep=r"\s+", header=None)
    z = mean.iloc[:,0].values
    z = np.array(z)
    redshifts = np.array(redshifts)
    
    mask = np.any(np.isclose(z[:, None], redshifts[None, :], atol=1e-8), axis=1)
    mask = ~mask

    new_mean = mean[mask]
    cov_DM_DH = np.loadtxt(folder + '/cov_DM_DH.txt')
    cov_alpha = np.loadtxt(folder + '/cov_alpha.txt')
    new_cov_DM_DH = cov_DM_DH[mask,:][:,mask]
    new_cov_alpha = cov_alpha[mask,:][:,mask]

    z = np.loadtxt(folder + '/redshifts.txt')
    if z.ndim == 0: z = np.array([z])
    mask = np.isin(z, redshifts, invert=True)
    new_z = z[mask]

    new_cov_DM_DH_DESIstyle = None
    new_mean_DESI_style = None
    if os.path.exists(folder + '/mean_LCDM_fid_DESIstyle.txt'):
        mean_DESI_style = pd.read_csv(folder + '/mean_LCDM_fid_DESIstyle.txt', sep=r"\s+", header=None)

        z = mean_DESI_style.iloc[:,0].values
        mask = np.isin(z, redshifts, invert=True)
        new_mean_DESI_style = mean_DESI_style[mask]
        cov_DM_DH_DESIstyle = np.loadtxt(folder + '/cov_DM_DH_DESIstyle.txt')
        new_cov_DM_DH_DESIstyle = cov_DM_DH_DESIstyle[mask,:][:,mask]

    return new_cov_DM_DH, new_cov_alpha, new_cov_DM_DH_DESIstyle, new_z, new_mean, new_mean_DESI_style

def save_temp_data(temp_folder, dat, overwrite=False):
    cDMDH, ca, c_DESI, z, m, m_DESI = dat
    os.makedirs(temp_folder, exist_ok=True)
    save_a_cov_mat(cDMDH, temp_folder, 'cov_DM_DH.txt', overwrite=overwrite)
    save_a_cov_mat(ca, temp_folder, 'cov_alpha.txt', overwrite=overwrite)
    if c_DESI is not None:
        save_a_cov_mat(c_DESI, temp_folder, 'cov_DM_DH_DESIstyle.txt', overwrite=overwrite)
    save_mean_data(m, temp_folder, 'mean_LCDM_fid.txt', overwrite=overwrite)
    if m_DESI is not None:
        save_mean_data(m_DESI, temp_folder, 'mean_LCDM_fid_DESIstyle.txt', overwrite=overwrite)
    np.savetxt(temp_folder + '/redshifts.txt', z, fmt="%.8e")

## removing redshifts:

In [ ]:
z_to_remove = [1.49]

path = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_splitBGS_LRG3ELG1_noLya'
cDMDH, ca, c_DESI, z, m, m_DESI = remove_z(path, z_to_remove)
dat = (cDMDH, ca, c_DESI, z, m, m_DESI)

In [ ]:
temp_folder = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/temporary_DESI_splitBGS_LRG3ELG1_noLya'

save_temp_data(temp_folder, dat, overwrite=False)

File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/temporary_DESI_splitBGS_LRG3ELG1_noLya/cov_DM_DH.txt
File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/temporary_DESI_splitBGS_LRG3ELG1_noLya/cov_alpha.txt
File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/temporary_DESI_splitBGS_LRG3ELG1_noLya/cov_DM_DH_DESIstyle.txt
File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/temporary_DESI_splitBGS_LRG3ELG1_noLya/mean_LCDM_fid.txt
File already exists: /home/adrien/PDM/code/PDM2026_wsl/cov_mat/temporary_DESI_splitBGS_LRG3ELG1_noLya/mean_LCDM_fid_DESIstyle.txt


## combine forecast

In [32]:
overwrite = False

In [34]:
fold1 = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_ext_noQSO'
fold2 = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_ext_QSO'

cov_DM_DH, cov_alpha, cov_DM_DH_DESIstyle, combined_z, combined_mean, combined_mean_DESI = combine_forecast([fold1, fold2])

newfold = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_ext_noLya'
if not os.path.exists(newfold):
    os.makedirs(newfold)

# save the combined covariance matrices and mean data
save_a_cov_mat(cov_DM_DH, newfold, '/cov_DM_DH.txt', overwrite=overwrite)
save_a_cov_mat(cov_alpha, newfold, '/cov_alpha.txt', overwrite=overwrite)
save_mean_data(combined_mean, newfold, '/mean_LCDM_fid.txt', overwrite=overwrite)
np.savetxt(newfold + '/redshifts.txt', combined_z, fmt="%.8e")

if cov_DM_DH_DESIstyle is not None:
    save_a_cov_mat(cov_DM_DH_DESIstyle, newfold, '/cov_DM_DH_DESIstyle.txt', overwrite=overwrite)

if combined_mean_DESI is not None:
    save_mean_data(combined_mean_DESI, newfold, '/mean_LCDM_fid_DESIstyle.txt', overwrite=overwrite)

In [58]:
print(pd.DataFrame(combined_mean))

        0          1           2
0   0.285   8.021697  DM_over_rs
1   0.285  26.004530  DH_over_rs
2   0.510  13.500537  DM_over_rs
3   0.510  22.739826  DH_over_rs
4   0.700  17.579508  DM_over_rs
5   0.700  20.243223  DH_over_rs
6   0.925  21.836231  DM_over_rs
7   0.925  17.663244  DH_over_rs
8   1.325  28.137752  DM_over_rs
9   1.325  14.033375  DH_over_rs
10  1.490  30.352565  DM_over_rs
11  1.490  12.838646  DH_over_rs


## add $\rm{Ly}\alpha$

In [4]:
def get_DESI_data():
    z_DESI = []
    path = r'/home/adrien/PDM/code/PDM2026_wsl/cobaya_packages/data/bao_data/desi_bao_dr2/desi_gaussian_bao_ALL_GCcomb_mean.txt'
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.split()
            z_DESI.append(float(parts[0]))

    z_DESI = set(z_DESI)
    z_DESI = list(z_DESI)
    z_DESI.sort()
    dict_distance = {}
    for z in z_DESI:
        dict_distance[z] = {}

    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.split()
            z = float(parts[0])
            for key in dict_distance.keys():
                if key == z:
                    dict_distance[key][parts[2]] = float(parts[1])
    
    return dict_distance, z_DESI, path

def getLya():
    path_covDESI = r'/home/adrien/PDM/code/PDM2026_wsl/cobaya_packages/data/bao_data/desi_bao_dr2/desi_gaussian_bao_ALL_GCcomb_cov.txt'
    covDESI = np.loadtxt(path_covDESI)
    covL = covDESI[-2:, -2:] # careful, Lya data is inverted !!!!
    covLya = covL.copy()
    covLya[0, 0] = covL[1, 1]
    covLya[1, 1] = covL[0, 0]
    covLya[0, 1] = covL[0, 1]
    covLya[1, 0] = covL[1, 0]
    _, z_DESI, _ = get_DESI_data() 
    zLya = z_DESI[-1]
    #get fiducial mean values for Lya
    meanLya = create_DESI_fid_data([zLya], DESI_style=False)

    covLya_alpha = covLya.copy()

    # covLya_alpha[0,0] = covLya[0,2*i]
    # covLya_alpha[2*i+1,2*i+1] = covLya[2*i+1,2*i+1]

    covLya_alpha[0,0] /= float(meanLya[0,1])**2
    covLya_alpha[1,1] /= float(meanLya[1,1])**2
    covLya_alpha[0,1] /= float(meanLya[0,1]) * float(meanLya[1,1])
    covLya_alpha[1,0] /= float(meanLya[1,1]) * float(meanLya[0,1])

    return zLya, covLya, covLya_alpha, meanLya

def scaleLya(Rfsky=None, divide_nbr=None):
    new_zLya, covLya, covLya_alpha, new_meanLya = getLya()

    rho = covLya_alpha[0,1] / np.sqrt(covLya_alpha[0,0] * covLya_alpha[1,1])

    scaled_covLya_alpha = covLya_alpha.copy()

    if Rfsky is not None:
        if Rfsky <= 0:
            raise ValueError("Rfsky must be positive.")
        else:
            scaled_covLya_alpha[0,0] *= 1/Rfsky # no sqrt since it's err^2
            scaled_covLya_alpha[1,1] *= 1/Rfsky
            scaled_covLya_alpha[0,1] = rho * np.sqrt(scaled_covLya_alpha[0,0] * scaled_covLya_alpha[1,1])
            scaled_covLya_alpha[1,0] = scaled_covLya_alpha[0,1]


    if divide_nbr is not None:
        if divide_nbr <= 1:
            raise ValueError("divide_nbr must be greater than 1.")

        # divide redshift bin by divide_nbr
        edges = np.linspace(1.8, 2.7, 1+divide_nbr)
        z_cent = 0.5 * (edges[:-1] + edges[1:])
        cosmo_planck = cosmoprimo.fiducial.DESI()
        bkg = cosmo_planck.get_background(engine="class")
        # get DC
        DC = bkg.comoving_angular_distance(edges)
        volume_ratios = []
        vini = (DC[-1]**3 - DC[0]**3)
        for i in range(len(DC)-1):
            volume_ratio = vini / (DC[i+1]**3 - DC[i]**3)
            volume_ratios.append(volume_ratio)
        print("volume_ratios = ", volume_ratios)
        
        new_scaled_covLya_alpha = np.zeros((2*divide_nbr, 2*divide_nbr))
        for i in range(divide_nbr):
            new_scaled_covLya_alpha[2*i, 2*i] = scaled_covLya_alpha[0,0] * volume_ratios[i]
            new_scaled_covLya_alpha[2*i+1, 2*i+1] = scaled_covLya_alpha[1,1] * volume_ratios[i]
            new_scaled_covLya_alpha[2*i, 2*i+1] = rho * np.sqrt(new_scaled_covLya_alpha[2*i, 2*i] * new_scaled_covLya_alpha[2*i+1, 2*i+1])
            new_scaled_covLya_alpha[2*i+1, 2*i] = new_scaled_covLya_alpha[2*i, 2*i+1]

        # mean and z
        new_zLya = z_cent
        new_meanLya = create_DESI_fid_data(new_zLya, DESI_style=False)
    else:
        new_scaled_covLya_alpha = scaled_covLya_alpha

    # mean and redshift change if divide_nbr is not None
    return new_zLya, new_scaled_covLya_alpha, new_meanLya

def addingLya(folder, Rfsky=None, divide_nbr=None, overwrite=False):
    # take last two rows and columns of covDESI

    zLya, covLya_alpha, meanLya = scaleLya(Rfsky, divide_nbr)

    covLya = covLya_alpha.copy()
    for i in range(covLya_alpha.shape[0]//2):
        covLya[2*i,2*i] *= float(meanLya[2*i,1])**2
        covLya[2*i+1,2*i+1] *= float(meanLya[2*i+1,1])**2
        covLya[2*i,2*i+1] *= float(meanLya[2*i,1]) * float(meanLya[2*i+1,1])
        covLya[2*i+1,2*i] *= float(meanLya[2*i+1,1]) * float(meanLya[2*i,1])

    cov = np.loadtxt(folder + '/cov_DM_DH.txt')
    cov_alpha = np.loadtxt(folder + '/cov_alpha.txt')
    cov_DESI = None
    if os.path.exists(folder + '/cov_DM_DH_DESIstyle.txt'):
        cov_DESI = np.loadtxt(folder + '/cov_DM_DH_DESIstyle.txt')  
    
    newcov = combine_cov_mat(cov, covLya)
    newcov_alpha = combine_cov_mat(cov_alpha, covLya_alpha)
    newcov_DESI = None
    if cov_DESI is not None:
        newcov_DESI = combine_cov_mat(cov_DESI, covLya)

    # combine redshifts
    z = np.loadtxt(folder + '/redshifts.txt')
    if z.ndim == 0: z = np.array([z])
    newz = np.append(z, zLya)

    mean = pd.read_csv(folder + '/mean_LCDM_fid.txt', sep=r"\s+", header=None)
    newmean = pd.concat([mean, pd.DataFrame(meanLya)], ignore_index=True)
    newmean_DESI = None
    if os.path.exists(folder + '/mean_LCDM_fid_DESIstyle.txt'):
        mean_DESI = pd.read_csv(folder + '/mean_LCDM_fid_DESIstyle.txt', sep=r"\s+", header=None)
        newmean_DESI = pd.concat([mean_DESI, pd.DataFrame(meanLya)], ignore_index=True)

    return newcov, newcov_alpha, newcov_DESI, newz, newmean, newmean_DESI

In [5]:
folder = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI_splitBGS_all'
newcov, newcov_alpha, newcov_DESI, newz, newmean, newmean_DESI = addingLya(folder, Rfsky=0.41/0.25)
newdat = (newcov, newcov_alpha, newcov_DESI, newz, newmean, newmean_DESI)

In [6]:
newfold = '/home/adrien/PDM/code/PDM2026_wsl/cov_mat//DESI_splitBGS_all_withLya'
save_temp_data(newfold, newdat, overwrite=False)

## DESI extension

In [ ]:
# Nominal program
# [z_center, sigma_DA/rd (%), sigma_H*rd (%)]
nominal = [
    # BGS BRIGHT
    [0.05,  6.65, 13.92],
    [0.15,  2.57,  5.40],
    [0.25,  1.64,  3.41],
    [0.35,  1.37,  2.70],
    # LRG
    [0.45,  1.25,  2.38],
    [0.55,  1.05,  1.99],
    [0.65,  0.92,  1.74],
    [0.75,  0.84,  1.56],
    [0.85,  0.78,  1.44],
    [0.95,  0.87,  1.52],
    [1.05,  1.25,  2.04],
    # ELG LOP
    [1.15,  1.24,  1.80],
    [1.25,  1.26,  1.80],
    [1.35,  1.30,  1.82],
    [1.45,  1.37,  1.89],
    [1.55,  1.87,  2.46],
    # Quasars
    [1.65,  3.39,  4.76],
    [1.75,  3.48,  4.87],
    [1.85,  3.67,  5.14],
    [1.95,  3.83,  5.36],
    [2.05,  4.22,  5.90],
    # Lya
    [2.15,  2.02,  2.16],
    [2.25,  2.14,  2.24],
    [2.35,  2.33,  2.36],
    [2.45,  2.56,  2.52],
    [2.55,  2.90,  2.77],
    [2.65,  3.38,  3.11],
    [2.75,  3.95,  3.50],
    [2.85,  4.69,  4.05],
    [2.95,  5.59,  4.71],
    [3.25,  4.39,  3.50],
]

# Extended program
extended = [
    # BGS BRIGHT
    [0.05,  6.03, 12.63],
    [0.15,  2.33,  4.90],
    [0.25,  1.49,  3.09],
    [0.35,  1.24,  2.45],
    # LRG
    [0.45,  1.04,  2.04],
    [0.55,  0.88,  1.71],
    [0.65,  0.77,  1.50],
    [0.75,  0.70,  1.33],
    [0.85,  0.65,  1.23],
    [0.95,  0.70,  1.26],
    [1.05,  0.93,  1.55],
    # ELG LOP
    [1.15,  1.03,  1.52],
    [1.25,  1.04,  1.52],
    [1.35,  1.07,  1.54],
    [1.45,  1.13,  1.59],
    [1.55,  1.51,  2.00],
    # Quasars
    [1.65,  3.08,  4.32],
    [1.75,  3.15,  4.42],
    [1.85,  3.33,  4.66],
    [1.95,  3.48,  4.86],
    [2.05,  3.83,  5.35],
    # Lya
    [2.15,  1.83,  1.96],
    [2.25,  1.94,  2.03],
    [2.35,  2.11,  2.14],
    [2.45,  2.32,  2.29],
    [2.55,  2.63,  2.51],
    [2.65,  3.06,  2.82],
    [2.75,  3.58,  3.18],
    [2.85,  4.25,  3.68],
    [2.95,  5.07,  4.27],
    [3.25,  4.00,  3.18],
]

nominal  = np.array(nominal)   # shape (31, 3)
extended = np.array(extended)  # shape (31, 3)

## Euclid

In [ ]:
# [z_center, f^P_Halpha, f^C_Halpha, n_true (h^3/Mpc^3), n_meas (h^3/Mpc^3)]
euclid_specs = [
    [1.0, 0.805, 0.387, 14.28e-4, 6.87e-4],
    [1.2, 0.796, 0.638,  9.49e-4, 7.61e-4],
    [1.4, 0.694, 0.724,  6.44e-4, 6.71e-4],
    [1.65, 0.879, 0.997, 4.03e-4, 4.57e-4],
]
euclid_specs = np.array(euclid_specs)

In [ ]:
n_obs = []

for i in range(4):
    ratio = euclid_specs[i,2]/euclid_specs[i,1]
    n_obs.append(ratio*euclid_specs[i,3])

In [12]:
for i in range(4):  
    print(f'n_obs = {n_obs[i]:.2e}')

n_obs = 6.87e-04
n_obs = 7.61e-04
n_obs = 6.72e-04
n_obs = 4.57e-04


In [16]:
def compute_Vshell(z_min, z_max, cosmo):
    r_min = cosmo.comoving_angular_distance(z_min)  # Mpc
    r_max = cosmo.comoving_angular_distance(z_max)  # Mpc
    Vshell = (4 * np.pi / 3) * (r_max**3 - r_min**3)
    return Vshell / 1e9 

In [17]:
cosmo = cosmoprimo.fiducial.DESI()


In [18]:
compute_Vshell(0.9, 1.1, cosmo)


np.float64(22.06156799176196)

In [19]:
v_meas = [7.03, 8.1, 8.9, 14.36]
v_tot = []
v_tot.append(compute_Vshell(0.9, 1.1, cosmo))
v_tot.append(compute_Vshell(1.1, 1.3, cosmo))
v_tot.append(compute_Vshell(1.3, 1.5, cosmo))
v_tot.append(compute_Vshell(1.5, 1.8, cosmo))

In [20]:
for i in range(len(v_meas)):
    print(v_meas[i]/v_tot[i])

0.31865368783511133
0.3182366641609018
0.31794855619632767
0.31766089724296015
